# Q14: PCA Implementation
**Topic:** Coding-question | **Difficulty:** Medium | **Time:** 20 min
**Sheet:** Grind75ML

---

## Question
Implement Principal Component Analysis (PCA) from scratch.

## Theory

**PCA** finds the directions (principal components) of maximum variance in the data.

**Steps:**
1. **Center** the data: $X' = X - \bar{X}$
2. **Compute covariance matrix:** $C = \frac{1}{n-1} X'^T X'$
3. **Eigendecomposition:** $C = V \Lambda V^T$
4. **Sort** eigenvectors by eigenvalue (descending)
5. **Project:** $X_{reduced} = X' \cdot V_k$ (top $k$ eigenvectors)

**Explained Variance Ratio:** $\frac{\lambda_i}{\sum_j \lambda_j}$ tells how much variance each component captures.

**Key Insight:** PCA = eigendecomposition of the covariance matrix = SVD of the centered data matrix.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


class PCA:
    """Principal Component Analysis from scratch."""
    
    def __init__(self, n_components: int):
        self.n_components = n_components
        self.components: np.ndarray = None  # principal axes (eigenvectors)
        self.eigenvalues: np.ndarray = None
        self.explained_variance_ratio: np.ndarray = None
        self.mean: np.ndarray = None
    
    def fit(self, X: np.ndarray) -> 'PCA':
        """Fit PCA on data X."""
        # Step 1: Center
        self.mean = np.mean(X, axis=0)
        X_centered = X - self.mean
        
        # Step 2: Covariance matrix
        n = X.shape[0]
        cov_matrix = (X_centered.T @ X_centered) / (n - 1)
        
        # Step 3: Eigendecomposition
        eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
        
        # Step 4: Sort by eigenvalue (descending)
        idx = np.argsort(eigenvalues)[::-1]
        eigenvalues = eigenvalues[idx]
        eigenvectors = eigenvectors[:, idx]
        
        # Keep top n_components
        self.eigenvalues = eigenvalues[:self.n_components]
        self.components = eigenvectors[:, :self.n_components]
        self.explained_variance_ratio = self.eigenvalues / np.sum(eigenvalues)
        return self
    
    def transform(self, X: np.ndarray) -> np.ndarray:
        """Project data onto principal components."""
        X_centered = X - self.mean
        return X_centered @ self.components
    
    def inverse_transform(self, X_reduced: np.ndarray) -> np.ndarray:
        """Reconstruct data from reduced representation."""
        return X_reduced @ self.components.T + self.mean
    
    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        return self.fit(X).transform(X)

In [ ]:
# --- Example: Reduce iris dataset from 4D to 2D ---
from sklearn.datasets import load_iris

iris = load_iris()
X = iris.data
y = iris.target

pca = PCA(n_components=2)
X_reduced = pca.fit_transform(X)

print(f"Original shape:  {X.shape}")
print(f"Reduced shape:   {X_reduced.shape}")
print(f"Explained variance: {pca.explained_variance_ratio}")
print(f"Total explained:    {sum(pca.explained_variance_ratio):.4f} ({sum(pca.explained_variance_ratio)*100:.1f}%)")

# Visualization
plt.figure(figsize=(8, 6))
for label, name in enumerate(iris.target_names):
    mask = y == label
    plt.scatter(X_reduced[mask, 0], X_reduced[mask, 1], label=name, alpha=0.7)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio[1]*100:.1f}% variance)')
plt.title('PCA: Iris Dataset (4D → 2D)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# --- Verify against sklearn ---
from sklearn.decomposition import PCA as SkPCA

sk_pca = SkPCA(n_components=2)
X_sk = sk_pca.fit_transform(X)

print(f"Our variance ratio:    {pca.explained_variance_ratio}")
print(f"Sklearn variance ratio: {sk_pca.explained_variance_ratio_}")
print(f"\nReconstruction error (ours):   {np.mean((X - pca.inverse_transform(X_reduced))**2):.6f}")
print(f"Reconstruction error (sklearn): {np.mean((X - sk_pca.inverse_transform(X_sk))**2):.6f}")

## Key Takeaways
- PCA is an **unsupervised** dimensionality reduction technique — it doesn't use labels
- Always **center** (and optionally scale) data before PCA
- Choose n_components by **explained variance ratio** (e.g., keep 95% of variance)
- Use **Incremental PCA** or **Randomized PCA** for large datasets that don't fit in memory